# CS 131 — Wildfire Burn Severity Mapping
## EDA & Data Pipeline Validation

This notebook covers:
1. GEE data export (run once)
2. SIFT alignment validation
3. dNBR computation and visualization
4. Class distribution across fires
5. Classical pipeline (Gaussian → Canny → Otsu) sanity check

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

from preprocess import load_tiff, sift_align, normalize_pair, preprocess_fire
from dnbr import compute_dnbr, dnbr_to_classes, otsu_threshold_dnbr, plot_dnbr_and_labels
from classical import run_classical_pipeline, evaluate, plot_classical_pipeline

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. GEE Export (run once)
Exports pre/post Landsat 8 pairs for all 5 fires to Google Drive.

In [ ]:
# from preprocess import export_all_fires
# export_all_fires()  # Uncomment to kick off GEE exports
# Check status at: https://code.earthengine.google.com/tasks

## 2. SIFT Alignment Validation
Visual check that pre/post image pairs are correctly aligned after SIFT homography.

In [ ]:
FIRE = 'camp_fire'

# Assumes raw tiffs downloaded from GDrive to data/raw/
pre_img, _ = load_tiff(f'../data/raw/{FIRE}_pre_2018.tif')
post_img, _ = load_tiff(f'../data/raw/{FIRE}_post_2018.tif')
post_aligned = sift_align(pre_img, post_img)
pre_norm, post_norm = normalize_pair(pre_img, post_aligned)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
def rgb(img): return np.clip(np.stack([img[:,:,2], img[:,:,1], img[:,:,0]], axis=2), 0, 1)

axes[0].imshow(rgb(pre_norm));  axes[0].set_title('Pre-fire'); axes[0].axis('off')
axes[1].imshow(rgb(post_norm)); axes[1].set_title('Post-fire (aligned)'); axes[1].axis('off')

# Difference image to check alignment quality
diff = np.abs(pre_norm[:,:,2] - post_norm[:,:,2])
axes[2].imshow(diff, cmap='hot'); axes[2].set_title('|Red band diff|'); axes[2].axis('off')
plt.suptitle(f'SIFT Alignment — {FIRE}', fontweight='bold')
plt.tight_layout()

## 3. dNBR Computation & Visualization

In [ ]:
dnbr, nbr_pre, nbr_post = compute_dnbr(pre_norm, post_norm)
labels_usgs = dnbr_to_classes(dnbr)
labels_otsu, thresholds = otsu_threshold_dnbr(dnbr)

print(f'dNBR range: [{dnbr.min():.3f}, {dnbr.max():.3f}]')
print(f'Otsu thresholds: {thresholds}')

plot_dnbr_and_labels(FIRE, pre_norm, post_norm, dnbr, labels_usgs,
                     title_suffix='USGS Thresholds',
                     save_path=f'../outputs/{FIRE}/dnbr_overview.png')

## 4. Class Distribution Across Fires

In [ ]:
from dnbr import CLASS_LABELS, CLASS_COLORS

fires = ['camp_fire', 'dixie_fire', 'caldor_fire', 'bootleg_fire', 'august_complex']
years = [2018, 2021, 2021, 2021, 2020]

fig, axes = plt.subplots(1, len(fires), figsize=(18, 4))
for ax, fire, year in zip(axes, fires, years):
    try:
        pre, _ = load_tiff(f'../data/processed/{fire}/pre.tif')
        post, _ = load_tiff(f'../data/processed/{fire}/post.tif')
        dnbr, _, _ = compute_dnbr(pre, post)
        labels = dnbr_to_classes(dnbr)
        counts = [(labels == i).mean() * 100 for i in range(4)]
        ax.bar(CLASS_LABELS, counts, color=CLASS_COLORS, edgecolor='black')
        ax.set_title(fire.replace('_', ' ').title())
        ax.set_ylabel('% pixels')
    except FileNotFoundError:
        ax.set_title(f'{fire}\n(not yet processed)')
plt.suptitle('Class Distribution by Fire', fontweight='bold')
plt.tight_layout()

## 5. Classical Pipeline (Gaussian → Canny → Otsu)

In [ ]:
dnbr_smooth, edges, pred_labels, thresholds = run_classical_pipeline(dnbr)
ious, acc = evaluate(pred_labels, labels_usgs)

plot_classical_pipeline(
    FIRE, dnbr, dnbr_smooth, edges, pred_labels, labels_usgs, ious,
    save_path=f'../outputs/{FIRE}/classical_pipeline.png'
)